In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

options = Options()
options.add_argument("--headless")

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
datos_totales = []
paginas_objetivo = 5

try: 
    driver.get("https://books.toscrape.com/")
    for i in range(paginas_objetivo):
        print(f"--- Procesando Pagina {i + 1} ---")
        
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod")))
        
        productos = driver.find_elements(By.CLASS_NAME, "product_pod")
        
        for p in productos:
            datos_totales.append(p.text)
            
        try:
            boton_siguiente = driver.find_element(By.XPATH, "//a[contains(text(),'next')]")
            boton_siguiente.click()
            time.sleep(3)
        except Exception:
            print("Se ha llegado a la ultima pagina o el boton no es visible.")
            break
    
    print(f"\nProceso finalizado con exito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error detectado durante la ejecucion: {e}")

finally:
    driver.quit()
    

--- Procesando Pagina 1 ---
--- Procesando Pagina 2 ---
--- Procesando Pagina 3 ---
--- Procesando Pagina 4 ---
--- Procesando Pagina 5 ---

Proceso finalizado con exito.
Total de registros capturados: 100


In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)



URL_OBJETIVO = "https://books.toscrape.com/" 

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = "article.product_pod" 
SELECTOR_DATO_A     = "h3 a"
SELECTOR_DATO_B     = "p.price_color"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) # Tiempo de espera para carga de datos dinámicos

    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            # Extracción genérica de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # Limpieza básica de caracteres no numéricos (opcional según el proyecto)
            valor_limpio = "".join(filter(str.isdigit, columna_b))

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0 # Indicador de registro procesado (float)
            })
        except:
           
            continue

        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head()) # Muestra de los primeros 5 datos
    else:
        print(" No se capturaron datos. Revisar los selectores CSS.")

finally:
    driver.quit()
    print(" Navegador cerrado.")

 Conectando a la fuente de datos...
 Se encontraron 20 registros potenciales.
 Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
      variable_nombre variable_valor  status
0  A Light in the ...           5177     1.0
 Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
      variable_nombre variable_valor  status
0  A Light in the ...           5177     1.0
1  Tipping the Velvet           5374     1.0
 Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
      variable_nombre variable_valor  status
0  A Light in the ...           5177     1.0
1  Tipping the Velvet           5374     1.0
2          Soumission           5010     1.0
 Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
      variable_nombre variable_valor  status
0  A Light in the ...           5177     1.0
1  Tipping the Velvet           5374     1.0
2          Soumission           5010     1.0
3       Sharp Objects           4782     1.0
 Proceso exitoso. Archivo generado: datos_ex